In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

base_path = Path().resolve().parent
file_path = base_path / 'data' / 'raw' / 'Stellar_edu_MDS_ap_stats_dataset - v1.9.xlsx'

In [2]:
data = pd.read_excel(file_path, sheet_name='Student_Observations')

In [3]:
data.head()

,student_id,assignment_id,class_num,observation_id,item_type,source_question,primary_kc_id,all_kc_ids,score,max_score,percent_score,student_response,correct_answer_or_rubric,rubric_level
0,S001,HW1,1,HW1_PCA_Q01,MCQ,PCA Q01,KC.U1.02.observational_unit_variable,KC.U1.02.observational_unit_variable,0.0,1,0,C,E,NaN
1,S001,HW1,1,HW1_PCA_Q02,MCQ,PCA Q02,KC.U1.03.variable_type_cat_quant,KC.U1.03.variable_type_cat_quant,1.0,1,100,E,E,NaN
2,S001,HW1,1,HW1_PCA_Q03,MCQ,PCA Q03,KC.U1.03.variable_type_cat_quant,KC.U1.03.variable_type_cat_quant,0.0,1,0,C,D,NaN
3,S001,HW1,1,HW1_PCA_Q04,MCQ,PCA Q04,KC.U1.05.categorical_freq_relative,KC.U1.05.categorical_freq_relative,1.0,1,100,E,E,NaN
4,S001,HW1,1,HW1_PCA_Q05,MCQ,PCA Q05,KC.U1.05.categorical_freq_relative,KC.U1.05.categorical_freq_relative,1.0,1,100,B,B,NaN


In [4]:
data['primary_kc_id'].nunique()

236

In [5]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 21808 entries, 0 to 21807
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   student_id                21808 non-null  str    
 1   assignment_id             21808 non-null  str    
 2   class_num                 21808 non-null  int64  
 3   observation_id            21808 non-null  str    
 4   item_type                 21808 non-null  str    
 5   source_question           21808 non-null  str    
 6   primary_kc_id             21808 non-null  str    
 7   all_kc_ids                21808 non-null  str    
 8   score                     21808 non-null  float64
 9   max_score                 21808 non-null  int64  
 10  percent_score             21808 non-null  int64  
 11  student_response          21808 non-null  str    
 12  correct_answer_or_rubric  21808 non-null  str    
 13  rubric_level              4642 non-null   str    
dtypes: float64(1), in

In [6]:
data.nunique()

student_id                   25
assignment_id                33
class_num                    27
observation_id              895
item_type                     3
source_question             640
primary_kc_id               236
all_kc_ids                  521
score                         3
max_score                     1
percent_score                 3
student_response             25
correct_answer_or_rubric    185
rubric_level                  3
dtype: int64

In [7]:
data[data['primary_kc_id']=='KC.U1.02.observational_unit_variable'].groupby('student_id').size().reset_index()

,student_id,0
0,S001,2
1,S002,2
2,S003,2
3,S004,2
4,S005,2
5,S006,2
6,S007,2
7,S008,2
8,S009,2
9,S010,2


In [8]:
data[data['primary_kc_id']=='KC.U1.05.categorical_freq_relative'].groupby('student_id').size().reset_index()

,student_id,0
0,S001,7
1,S002,7
2,S003,7
3,S004,7
4,S005,7
5,S006,6
6,S007,7
7,S008,7
8,S009,7
9,S010,7


**The Criteria for the "Best Subset"**
To make BKT work, a Knowledge Component (KC) must meet three strict criteria:

- Vertical Depth (Sequence): Students must have attempted the skill at least 3+ times on average. BKT cannot "see" learning if there is only one data point per person.

- Horizontal Breadth (Reach): At least 5–10 students should have attempted the skill. If only one student did a KC, the model will "overfit" to that one person's specific mistakes.

- Statistical Variance: The KC cannot have 100% accuracy or 0% accuracy. The model needs to see the "flip" from incorrect to correct to calculate the Learn Rate.

In [9]:
data.columns

Index(['student_id', 'assignment_id', 'class_num', 'observation_id',
       'item_type', 'source_question', 'primary_kc_id', 'all_kc_ids', 'score',
       'max_score', 'percent_score', 'student_response',
       'correct_answer_or_rubric', 'rubric_level'],
      dtype='str')

In [10]:
# check to select KC's that were attempted by at least 3 times per student

check_1 = data.groupby(['primary_kc_id', 'student_id']).size().reset_index(name='attempts')


check_1

,primary_kc_id,student_id,attempts
0,KC.U1.02.observational_unit_variable,S001,2
1,KC.U1.02.observational_unit_variable,S002,2
2,KC.U1.02.observational_unit_variable,S003,2
3,KC.U1.02.observational_unit_variable,S004,2
4,KC.U1.02.observational_unit_variable,S005,2
...,...,...,...
5824,KC.U9.20.regression_inference_scope_and_predic...,S021,2
5825,KC.U9.20.regression_inference_scope_and_predic...,S022,2
5826,KC.U9.20.regression_inference_scope_and_predic...,S023,2
5827,KC.U9.20.regression_inference_scope_and_predic...,S024,2


In [11]:
check_1['primary_kc_id'].nunique()

236

In [12]:
check_1.attempts.value_counts(normalize=True) *100


attempts
2     21.770458
3     21.358724
4     13.141191
1     13.021101
5     10.893807
6      8.217533
7      5.781438
8      2.882141
9      1.646938
13     0.428890
14     0.411734
17     0.411734
12     0.017156
15     0.017156
Name: proportion, dtype: float64

In [13]:
check_1[check_1['attempts'] == 2]['primary_kc_id'].nunique()

91

In [14]:
check_1.groupby('primary_kc_id')['attempts'].min()

primary_kc_id
KC.U1.02.observational_unit_variable                           2
KC.U1.03.variable_type_cat_quant                               3
KC.U1.04.variable_type_discrete_continuous                     3
KC.U1.05.categorical_freq_relative                             6
KC.U1.06.categorical_bar_pie_segmented                         7
                                                              ..
KC.U9.16.slope_test_pvalue_interpret                           1
KC.U9.17.slope_test_decision_conclusion                        1
KC.U9.18.scatterplot_for_regression_inference                  2
KC.U9.19.lsrl_equation_from_data                               2
KC.U9.20.regression_inference_scope_and_prediction_language    2
Name: attempts, Length: 236, dtype: int64

In [15]:
# the minimum number of attempts any student had for each KC

kc_min_attempts = (
    check_1.groupby('primary_kc_id')['attempts']
    .min()
    .reset_index()
    .rename(columns={'attempts': 'min_attempts'})
)

print(kc_min_attempts['min_attempts'].value_counts().sort_index())
print(f"KCs where ALL students have >= 3 attempts: {(kc_min_attempts['min_attempts'] >= 3).sum()}")
print(f"KCs where at least one student has only 1:  {(kc_min_attempts['min_attempts'] == 1).sum()}")

min_attempts
1     83
2     59
3     33
4     18
5     18
6     11
7      7
8      3
9      1
12     1
13     1
15     1
Name: count, dtype: int64
KCs where ALL students have >= 3 attempts: 94
KCs where at least one student has only 1:  83


In [16]:
greater_3_stud = check_1[check_1['attempts'] >= 3]
greater_3_stud

,primary_kc_id,student_id,attempts
25,KC.U1.03.variable_type_cat_quant,S001,3
26,KC.U1.03.variable_type_cat_quant,S002,3
27,KC.U1.03.variable_type_cat_quant,S003,3
28,KC.U1.03.variable_type_cat_quant,S004,3
29,KC.U1.03.variable_type_cat_quant,S005,3
...,...,...,...
5800,KC.U9.19.lsrl_equation_from_data,S021,3
5801,KC.U9.19.lsrl_equation_from_data,S022,3
5802,KC.U9.19.lsrl_equation_from_data,S023,3
5803,KC.U9.19.lsrl_equation_from_data,S024,3


In [17]:
greater_3_stud['primary_kc_id'].nunique()

157

In [18]:
# filter to see KCs that were done by a minimum of 5 students

greater_3_stud.groupby('primary_kc_id').filter(lambda x: len(x) >= 5)

,primary_kc_id,student_id,attempts
25,KC.U1.03.variable_type_cat_quant,S001,3
26,KC.U1.03.variable_type_cat_quant,S002,3
27,KC.U1.03.variable_type_cat_quant,S003,3
28,KC.U1.03.variable_type_cat_quant,S004,3
29,KC.U1.03.variable_type_cat_quant,S005,3
...,...,...,...
5800,KC.U9.19.lsrl_equation_from_data,S021,3
5801,KC.U9.19.lsrl_equation_from_data,S022,3
5802,KC.U9.19.lsrl_equation_from_data,S023,3
5803,KC.U9.19.lsrl_equation_from_data,S024,3


In [19]:
greater_3_stud.groupby('primary_kc_id').size().sort_values(ascending=True)

primary_kc_id
KC.U6.10.confidence_level_long_run_interpretation    21
KC.U7.07.ci_width_sample_size_confidence             21
KC.U9.19.lsrl_equation_from_data                     22
KC.U6.08.margin_error_width_sample_size              22
KC.U10.05.bootstrap_percentile_cutoffs               22
                                                     ..
KC.U3.16.explanatory_response_treatments             25
KC.U3.13.stratification_design_advantage             25
KC.U3.11.generalization_scope                        25
KC.U4.11.independent_events_probability              25
KC.U4.32.probability_from_simulation                 25
Length: 157, dtype: int64

In [20]:
# sub-setting the data

cleaned_1_data = data[data['primary_kc_id'].isin(greater_3_stud['primary_kc_id'].unique())].reset_index(drop=True)
cleaned_1_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 18679 entries, 0 to 18678
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   student_id                18679 non-null  str    
 1   assignment_id             18679 non-null  str    
 2   class_num                 18679 non-null  int64  
 3   observation_id            18679 non-null  str    
 4   item_type                 18679 non-null  str    
 5   source_question           18679 non-null  str    
 6   primary_kc_id             18679 non-null  str    
 7   all_kc_ids                18679 non-null  str    
 8   score                     18679 non-null  float64
 9   max_score                 18679 non-null  int64  
 10  percent_score             18679 non-null  int64  
 11  student_response          18679 non-null  str    
 12  correct_answer_or_rubric  18679 non-null  str    
 13  rubric_level              4046 non-null   str    
dtypes: float64(1), in

In [21]:
cleaned_1_data['primary_kc_id'].nunique()

157

Other checks regarding data suitability;

In [22]:
kc_stats = cleaned_1_data.groupby('primary_kc_id').agg(
    total_no_obs=('score', 'count'),
    unique_students=('student_id', 'nunique'),
    avg_score= ('score', 'mean'),
    avg_std=('score', 'std')
)

kc_stats.sort_values(by='total_no_obs')

,total_no_obs,unique_students,avg_score,avg_std
primary_kc_id,,,,
KC.U10.11.cohens_d_interpret_practical_importance,69,23,0.500000,0.469668
KC.U8.21.two_way_p_value_interpretation,69,23,0.420290,0.497222
KC.U7.04.one_sample_t_interval_conditions,69,23,0.405797,0.463961
KC.U6.10.confidence_level_long_run_interpretation,69,25,0.449275,0.501065
KC.U10.07.bootstrap_interval_interpret_context,70,25,0.400000,0.413539
...,...,...,...,...
KC.U4.01.probability_long_run_interpretation,224,25,0.633929,0.482808
KC.U2.03.conditional_relative_frequency,225,25,0.577778,0.483610
KC.U4.11.independent_events_probability,324,25,0.561728,0.496942


In [23]:
kc_stats.describe()

,total_no_obs,unique_students,avg_score,avg_std
count,157.000000,157.000000,157.000000,157.000000
mean,118.974522,24.904459,0.523809,0.478435
std,54.063584,0.371819,0.072336,0.025448
min,69.000000,23.000000,0.340278,0.406748
25%,74.000000,25.000000,0.479885,0.466486
50%,100.000000,25.000000,0.523196,0.484377
75%,148.000000,25.000000,0.565714,0.500000
max,423.000000,25.000000,0.729167,0.503413


In [24]:
min_obs = 30
# min_students = 10
min_var = 0.1

kc_stats[
    (kc_stats['total_no_obs']>=min_obs) &
    #(kc_stats['unique_students']>=min_students) &
    (kc_stats['avg_std']>min_var)
]

,total_no_obs,unique_students,avg_score,avg_std
primary_kc_id,,,,
KC.U1.03.variable_type_cat_quant,75,25,0.666667,0.474579
KC.U1.04.variable_type_discrete_continuous,99,25,0.626263,0.486257
KC.U1.05.categorical_freq_relative,174,25,0.597701,0.491777
KC.U1.06.categorical_bar_pie_segmented,175,25,0.657143,0.476026
KC.U1.07.two_way_table_association,75,25,0.560000,0.499730
...,...,...,...,...
KC.U9.14.slope_test_hypotheses,94,25,0.563830,0.441369
KC.U9.16.slope_test_pvalue_interpret,72,25,0.402778,0.493899
KC.U9.17.slope_test_decision_conclusion,96,25,0.458333,0.473879


In [25]:
cleaned_1_data[cleaned_1_data['primary_kc_id'].isin(kc_stats.index.to_list())]

,student_id,assignment_id,class_num,observation_id,item_type,source_question,primary_kc_id,all_kc_ids,score,max_score,percent_score,student_response,correct_answer_or_rubric,rubric_level
0,S001,HW1,1,HW1_PCA_Q02,MCQ,PCA Q02,KC.U1.03.variable_type_cat_quant,KC.U1.03.variable_type_cat_quant,1.0,1,100,E,E,NaN
1,S001,HW1,1,HW1_PCA_Q03,MCQ,PCA Q03,KC.U1.03.variable_type_cat_quant,KC.U1.03.variable_type_cat_quant,0.0,1,0,C,D,NaN
2,S001,HW1,1,HW1_PCA_Q04,MCQ,PCA Q04,KC.U1.05.categorical_freq_relative,KC.U1.05.categorical_freq_relative,1.0,1,100,E,E,NaN
3,S001,HW1,1,HW1_PCA_Q05,MCQ,PCA Q05,KC.U1.05.categorical_freq_relative,KC.U1.05.categorical_freq_relative,1.0,1,100,B,B,NaN
4,S001,HW1,1,HW1_PCA_Q06,MCQ,PCA Q06,KC.U1.05.categorical_freq_relative,KC.U1.05.categorical_freq_relative,1.0,1,100,D,D,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18674,S025,MF2,27,MF2_FRQ5C,FRQ_Component,FRQ 1,KC.U9.17.slope_test_decision_conclusion,KC.U9.17.slope_test_decision_conclusion|KC.U9....,0.0,1,0,I,Rubric: use test statistic/p-value and conclud...,I
18675,S025,MF2,27,MF2_FRQ6B,FRQ_Component,FRQ 6,KC.U7.12.mean_hypotheses,KC.U7.12.mean_hypotheses|KC.U7.11.matched_pair...,1.0,1,100,E,Rubric: define the population mean paired diff...,E
18676,S025,MF2,27,MF2_FRQ6C,FRQ_Component,FRQ 6,KC.U7.13.t_test_conditions,KC.U7.13.t_test_conditions|KC.U7.11.matched_pa...,0.5,1,50,P,"Rubric: address paired structure, random sampl...",P
18677,S025,MF2,27,MF2_FRQ6D,FRQ_Component,FRQ 6,KC.U7.14.t_test_statistic_mean,KC.U7.14.t_test_statistic_mean|KC.U7.15.p_valu...,1.0,1,100,E,"Rubric: use mean difference, standard error, d...",E


In [26]:
cleaned_1_data.info()


<class 'pandas.DataFrame'>
RangeIndex: 18679 entries, 0 to 18678
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   student_id                18679 non-null  str    
 1   assignment_id             18679 non-null  str    
 2   class_num                 18679 non-null  int64  
 3   observation_id            18679 non-null  str    
 4   item_type                 18679 non-null  str    
 5   source_question           18679 non-null  str    
 6   primary_kc_id             18679 non-null  str    
 7   all_kc_ids                18679 non-null  str    
 8   score                     18679 non-null  float64
 9   max_score                 18679 non-null  int64  
 10  percent_score             18679 non-null  int64  
 11  student_response          18679 non-null  str    
 12  correct_answer_or_rubric  18679 non-null  str    
 13  rubric_level              4046 non-null   str    
dtypes: float64(1), in

In [27]:
cleaned_1_data.to_csv(base_path / 'data' / 'processed' / 'best_subset_1_data.csv', index=False)

In [28]:
cleaned_1_data.shape

(18679, 14)

In [29]:
data.shape

(21808, 14)

## KC EDGES: Re-Grouping

In [30]:
edges = pd.read_excel(file_path, sheet_name='KC_Edges')

In [31]:
edges.info()

<class 'pandas.DataFrame'>
RangeIndex: 342 entries, 0 to 341
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   edge_id        342 non-null    str    
 1   source_kc_id   342 non-null    str    
 2   target_kc_id   342 non-null    str    
 3   relation       342 non-null    str    
 4   official       342 non-null    bool   
 5   confidence     342 non-null    float64
 6   evidence_note  342 non-null    str    
dtypes: bool(1), float64(1), str(5)
memory usage: 16.5 KB


In [32]:
edges['source_kc_id'].unique()

<StringArray>
[                     'KC.U1.01.statistical_context',
              'KC.U1.02.observational_unit_variable',
                  'KC.U1.03.variable_type_cat_quant',
                'KC.U1.05.categorical_freq_relative',
            'KC.U1.06.categorical_bar_pie_segmented',
                'KC.U1.07.two_way_table_association',
        'KC.U1.04.variable_type_discrete_continuous',
             'KC.U1.09.quantitative_display_reading',
                 'KC.U1.10.histogram_shape_features',
                  'KC.U1.12.center_mean_median_mode',
 ...
              'KC.U10.03.bootstrap_same_sample_size',
     'KC.U10.04.bootstrap_distribution_of_statistic',
            'KC.U10.05.bootstrap_percentile_cutoffs',
 'KC.U10.06.bootstrap_percentile_interval_construct',
            'KC.U10.10.cohens_d_compute_effect_size',
         'KC.U10.13.type_ii_error_context_extension',
       'KC.U10.14.power_beta_relationship_extension',
      'KC.U10.15.rejection_region_power_calculation',
         

In [33]:
edges[edges['source_kc_id'].duplicated(keep=False)]

,edge_id,source_kc_id,target_kc_id,relation,official,confidence,evidence_note
0,E001,KC.U1.01.statistical_context,KC.U1.02.observational_unit_variable,supports,False,0.80,Context helps identify what is measured.
2,E003,KC.U1.03.variable_type_cat_quant,KC.U1.04.variable_type_discrete_continuous,prerequisite,False,0.85,Discrete/continuous only applies to quantitati...
3,E004,KC.U1.03.variable_type_cat_quant,KC.U1.05.categorical_freq_relative,supports,False,0.75,Categorical variables are summarized in tables.
4,E005,KC.U1.05.categorical_freq_relative,KC.U1.06.categorical_bar_pie_segmented,supports,False,0.80,Tables support interpreting categorical graphs.
5,E006,KC.U1.05.categorical_freq_relative,KC.U1.07.two_way_table_association,supports,False,0.80,Association uses conditional counts/proportions.
...,...,...,...,...,...,...,...
334,E335,KC.U6.20.significance_level_alpha,KC.U10.15.rejection_region_power_calculation,supports,False,0.80,Alpha determines the rejection region used for...
335,E336,KC.U5.20.sample_mean_probability,KC.U10.15.rejection_region_power_calculation,supports,False,0.75,Power calculations often use normal probabilit...
337,E338,KC.U5.05.sample_size_effect_variability,KC.U10.16.factors_affecting_power,supports,False,0.80,Sample size and variability affect power throu...
339,E340,KC.U10.16.factors_affecting_power,KC.U10.17.power_specific_alternative,supports,False,0.75,Effect size and true alternative value affect ...


In [34]:
data_2 = data.copy()

In [35]:
# if KCs are prerequisites of the other, we could group them as one KC

kc_groups = {
    "KC.U1.02.observational_unit_variable": "KC.U1.02.03.04",
    "KC.U1.03.variable_type_cat_quant": "KC.U1.02.03.04",
    "KC.U1.04.variable_type_discrete_continuous": "KC.U1.02.03.04",

    "KC.U1.09.quantitative_display_reading": "KC.U1.09.10",
    "KC.U1.10.histogram_shape_features": "KC.U1.09.10",

    "KC.U1.14.five_number_boxplot": "KC.U1.14.15",
    "KC.U1.15.outlier_rule": "KC.U1.14.15",

    "KC.U1.19.z_score_compute_interpret": "KC.U1.19.22",
    "KC.U1.22.normal_tail_interval_probability": "KC.U1.19.22",

    "KC.U2.01.two_way_table_structure": "KC.U2.01.03",
    "KC.U2.03.conditional_relative_frequency": "KC.U2.01.03",

    "KC.U2.06.scatterplot_construction": "KC.U2.06.07.09.10",
    "KC.U2.07.scatterplot_describe_association": "KC.U2.06.07.09.10",
    "KC.U2.09.correlation_properties": "KC.U2.06.07.09.10",
    "KC.U2.10.correlation_interpretation": "KC.U2.06.07.09.10",

    "KC.U3.03.simple_random_sample": "KC.U3.03.04.13",
    "KC.U3.04.stratified_random_sample": "KC.U3.03.04.13",
    "KC.U3.13.stratification_design_advantage": "KC.U3.03.04.13",

    "KC.U3.17.random_assignment": "KC.U3.17.18",
    "KC.U3.18.causation_from_experiment": "KC.U3.17.18",

    "KC.U4.03.sample_space_outcomes": "KC.U4.03.04",
    "KC.U4.04.basic_probability_counting": "KC.U4.03.04",

    "KC.U4.14.random_variable_definition": "KC.U4.14.15",
    "KC.U4.15.discrete_probability_distribution": "KC.U4.14.15",

    "KC.U4.23.binomial_conditions": "KC.U4.23.24",
    "KC.U4.24.binomial_exact_probability": "KC.U4.23.24",

    "KC.U4.28.geometric_conditions": "KC.U4.28.29",
    "KC.U4.29.geometric_probability": "KC.U4.28.29",

    "KC.U5.06.point_estimator_statistic_parameter": "KC.U5.06.07",
    "KC.U5.07.unbiased_estimator": "KC.U5.06.07",

    "KC.U5.10.sample_proportion_standard_error": "KC.U5.10.11.12",
    "KC.U5.11.sample_proportion_normal_conditions": "KC.U5.10.11.12",
    "KC.U5.12.sample_proportion_probability": "KC.U5.10.11.12",

    "KC.U5.14.difference_sample_proportions_standard_error": "KC.U5.14.15.16",
    "KC.U5.15.difference_sample_proportions_normal_conditions": "KC.U5.14.15.16",
    "KC.U5.16.difference_sample_proportions_probability": "KC.U5.14.15.16",

    "KC.U5.17.sample_mean_mean": "KC.U5.17.18.19.20",
    "KC.U5.18.sample_mean_standard_error": "KC.U5.17.18.19.20",
    "KC.U5.19.clt_sample_mean_shape": "KC.U5.17.18.19.20",
    "KC.U5.20.sample_mean_probability": "KC.U5.17.18.19.20",

    "KC.U6.03.sample_proportion_se_for_inference": "KC.U6.03.04.05.06",
    "KC.U6.04.one_prop_ci_conditions": "KC.U6.03.04.05.06",
    "KC.U6.05.critical_value_confidence_level": "KC.U6.03.04.05.06",
    "KC.U6.06.one_prop_ci_compute": "KC.U6.03.04.05.06",

    "KC.U6.13.one_prop_test_conditions": "KC.U6.13.14",
    "KC.U6.14.one_prop_test_statistic": "KC.U6.13.14",

    "KC.U6.25.two_prop_ci_standard_error": "KC.U6.25.26",
    "KC.U6.26.two_prop_ci_compute_interpret": "KC.U6.25.26",

    "KC.U6.31.two_prop_test_statistic_pvalue": "KC.U6.31.32",
    "KC.U6.32.two_prop_test_conclusion": "KC.U6.31.32",

    "KC.U7.04.one_sample_t_interval_conditions": "KC.U7.04.05.06.07",
    "KC.U7.05.one_sample_t_interval_compute": "KC.U7.04.05.06.07",
    "KC.U7.06.t_interval_interpretation": "KC.U7.04.05.06.07",

    "KC.U7.12.mean_hypotheses": "KC.U7.12.13.14.15.16",
    "KC.U7.13.t_test_conditions": "KC.U7.12.13.14.15.16",
    "KC.U7.14.t_test_statistic_mean": "KC.U7.12.13.14.15.16",
    "KC.U7.15.p_value_interpretation_mean": "KC.U7.12.13.14.15.16",
    "KC.U7.16.t_test_decision_conclusion": "KC.U7.12.13.14.15.16",

    "KC.U7.18.two_sample_t_interval_procedure_selection": "KC.U7.18.19.20",
    "KC.U7.19.two_sample_t_interval_conditions": "KC.U7.18.19.20",
    "KC.U7.20.two_sample_t_interval_compute_interpret": "KC.U7.18.19.20",

    "KC.U7.24.two_sample_t_test_statistic_pvalue": "KC.U7.24.25",
    "KC.U7.25.two_sample_t_test_conclusion": "KC.U7.24.25",
}


# {
#     'KC.U1.02.03.04': 'KC.U1.02.observational_unit_variable',
#     'KC.U1.02.03.04': 'KC.U1.03.variable_type_cat_quant',
#     'KC.U1.02.03.04': 'KC.U1.04.variable_type_discrete_continuous',
    
#     'KC.U1.09.10': 'KC.U1.09.quantitative_display_reading',
#     'KC.U1.09.10': 'KC.U1.10.histogram_shape_features',
    
#     'KC.U1.14.15': 'KC.U1.14.five_number_boxplot',
#     'KC.U1.14.15': 'KC.U1.15.outlier_rule',
    
#     'KC.U1.19.22': 'KC.U1.19.z_score_compute_interpret',
#     'KC.U1.19.22': 'KC.U1.22.normal_tail_interval_probability',
    
#     'U1.03': 'Data_Types',
#     'U1.04': 'Data_Types',
#     'U1.05': 'Distribution_Shape',
#     'U1.17': 'Distribution_Shape',
#     'U1.18': 'Distribution_Shape',
#     'U1.10': 'Measures_of_Center',
#     'U1.11': 'Measures_of_Center',
#     'U1.02': 'Data_Types',
#     'U1.03': 'Data_Types',
#     'U1.04': 'Data_Types',    
#     'U1.02': 'Data_Types',
#     'U1.03': 'Data_Types',
#     'U1.04': 'Data_Types',
# }

In [36]:
data_2['primary_kc_id'] = data_2['primary_kc_id'].map(kc_groups).fillna(data_2['primary_kc_id'])

In [37]:
check_2 = data_2.groupby(['primary_kc_id', 'student_id']).size().reset_index(name='attempts')
check_2

,primary_kc_id,student_id,attempts
0,KC.U1.02.03.04,S001,9
1,KC.U1.02.03.04,S002,9
2,KC.U1.02.03.04,S003,9
3,KC.U1.02.03.04,S004,9
4,KC.U1.02.03.04,S005,9
...,...,...,...
5008,KC.U9.20.regression_inference_scope_and_predic...,S021,2
5009,KC.U9.20.regression_inference_scope_and_predic...,S022,2
5010,KC.U9.20.regression_inference_scope_and_predic...,S023,2
5011,KC.U9.20.regression_inference_scope_and_predic...,S024,2


In [50]:
check_2.attempts.value_counts(normalize=True) *100

attempts
2    22.68103
3    20.28725
1    12.54738
4     9.47536
5     9.29583
6     6.82226
7     5.88470
9     3.29144
8     2.45362
13    1.95492
10    1.39637
17    0.97746
11    0.97746
14    0.93756
15    0.43886
27    0.41891
12    0.07979
18    0.01995
22    0.01995
26    0.01995
23    0.01995
Name: proportion, dtype: float64

In [39]:
kc_min_attempts2 = (
    check_2.groupby('primary_kc_id')['attempts']
    .min()
    .reset_index()
    .rename(columns={'attempts': 'min_attempts'})
)

print(kc_min_attempts2['min_attempts'].value_counts().sort_index())
print(f"KCs where ALL students have >= 3 attempts: {(kc_min_attempts2['min_attempts'] >= 3).sum()}")
print(f"KCs where at least one student has only 1:  {(kc_min_attempts2['min_attempts'] == 1).sum()}")

min_attempts
1     72
2     49
3     22
4     11
5     16
6      9
7      9
8      4
9      1
10     2
11     1
12     3
13     1
15     1
17     1
18     1
Name: count, dtype: int64
KCs where ALL students have >= 3 attempts: 82
KCs where at least one student has only 1:  72


In [40]:
greater_3_stud_2 = check_2[check_2['attempts'] >= 3]
greater_3_stud_2

,primary_kc_id,student_id,attempts
0,KC.U1.02.03.04,S001,9
1,KC.U1.02.03.04,S002,9
2,KC.U1.02.03.04,S003,9
3,KC.U1.02.03.04,S004,9
4,KC.U1.02.03.04,S005,9
...,...,...,...
4984,KC.U9.19.lsrl_equation_from_data,S021,3
4985,KC.U9.19.lsrl_equation_from_data,S022,3
4986,KC.U9.19.lsrl_equation_from_data,S023,3
4987,KC.U9.19.lsrl_equation_from_data,S024,3


In [41]:
greater_3_stud_2.groupby('primary_kc_id').size().sort_values(ascending=True)

primary_kc_id
KC.U7.07.ci_width_sample_size_confidence             21
KC.U6.10.confidence_level_long_run_interpretation    21
KC.U6.27.two_prop_ci_plausibility_difference         22
KC.U6.24.two_prop_inference_conditions               22
KC.U10.07.bootstrap_interval_interpret_context       22
                                                     ..
KC.U3.19.control_comparison_replication              25
KC.U3.17.18                                          25
KC.U3.16.explanatory_response_treatments             25
KC.U3.10.response_wording_bias                       25
KC.U4.25.binomial_cumulative_probability             25
Length: 134, dtype: int64

In [42]:
# sub-setting the data

cleaned_2_data = data_2[data_2['primary_kc_id'].isin(greater_3_stud_2['primary_kc_id'].unique())].reset_index(drop=True)
cleaned_2_data.info()

cleaned_2_data['primary_kc_id'].nunique()

<class 'pandas.DataFrame'>
RangeIndex: 19043 entries, 0 to 19042
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   student_id                19043 non-null  str    
 1   assignment_id             19043 non-null  str    
 2   class_num                 19043 non-null  int64  
 3   observation_id            19043 non-null  str    
 4   item_type                 19043 non-null  str    
 5   source_question           19043 non-null  str    
 6   primary_kc_id             19043 non-null  str    
 7   all_kc_ids                19043 non-null  str    
 8   score                     19043 non-null  float64
 9   max_score                 19043 non-null  int64  
 10  percent_score             19043 non-null  int64  
 11  student_response          19043 non-null  str    
 12  correct_answer_or_rubric  19043 non-null  str    
 13  rubric_level              4191 non-null   str    
dtypes: float64(1), in

134

In [43]:
kc_stats_2 = cleaned_2_data.groupby('primary_kc_id').agg(
    total_no_obs=('score', 'count'),
    unique_students=('student_id', 'nunique'),
    avg_score= ('score', 'mean'),
    avg_std=('score', 'std')
)

kc_stats_2.sort_values(by='total_no_obs')

,total_no_obs,unique_students,avg_score,avg_std
primary_kc_id,,,,
KC.U6.10.confidence_level_long_run_interpretation,69,25,0.449275,0.501065
KC.U10.11.cohens_d_interpret_practical_importance,69,23,0.500000,0.469668
KC.U8.21.two_way_p_value_interpretation,69,23,0.420290,0.497222
KC.U6.27.two_prop_ci_plausibility_difference,70,25,0.542857,0.501757
KC.U6.08.margin_error_width_sample_size,70,25,0.457143,0.501757
...,...,...,...,...
KC.U3.11.generalization_scope,349,25,0.568768,0.469928
KC.U5.14.15.16,356,25,0.484551,0.449303
KC.U4.16.expected_value_calculation,423,25,0.557920,0.466486


In [44]:
min_obs = 30
# min_students = 10
min_var = 0.1

kc_stats_2[
    (kc_stats_2['total_no_obs']>=min_obs) &
    #(kc_stats_2['unique_students']>=min_students) &
    (kc_stats_2['avg_std']>min_var)
]

,total_no_obs,unique_students,avg_score,avg_std
primary_kc_id,,,,
KC.U1.02.03.04,224,25,0.647321,0.478874
KC.U1.05.categorical_freq_relative,174,25,0.597701,0.491777
KC.U1.06.categorical_bar_pie_segmented,175,25,0.657143,0.476026
KC.U1.07.two_way_table_association,75,25,0.560000,0.499730
KC.U1.09.10,222,25,0.563063,0.483282
...,...,...,...,...
KC.U9.14.slope_test_hypotheses,94,25,0.563830,0.441369
KC.U9.16.slope_test_pvalue_interpret,72,25,0.402778,0.493899
KC.U9.17.slope_test_decision_conclusion,96,25,0.458333,0.473879


In [45]:
cleaned_2_data[cleaned_2_data['primary_kc_id'].isin(kc_stats_2.index.to_list())]

#cleaned_2_data.to_csv(base_path / 'data' / 'processed' / 'best_subset_2_data.csv', index=False)

,student_id,assignment_id,class_num,observation_id,item_type,source_question,primary_kc_id,all_kc_ids,score,max_score,percent_score,student_response,correct_answer_or_rubric,rubric_level
0,S001,HW1,1,HW1_PCA_Q01,MCQ,PCA Q01,KC.U1.02.03.04,KC.U1.02.observational_unit_variable,0.0,1,0,C,E,NaN
1,S001,HW1,1,HW1_PCA_Q02,MCQ,PCA Q02,KC.U1.02.03.04,KC.U1.03.variable_type_cat_quant,1.0,1,100,E,E,NaN
2,S001,HW1,1,HW1_PCA_Q03,MCQ,PCA Q03,KC.U1.02.03.04,KC.U1.03.variable_type_cat_quant,0.0,1,0,C,D,NaN
3,S001,HW1,1,HW1_PCA_Q04,MCQ,PCA Q04,KC.U1.05.categorical_freq_relative,KC.U1.05.categorical_freq_relative,1.0,1,100,E,E,NaN
4,S001,HW1,1,HW1_PCA_Q05,MCQ,PCA Q05,KC.U1.05.categorical_freq_relative,KC.U1.05.categorical_freq_relative,1.0,1,100,B,B,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19038,S025,MF2,27,MF2_FRQ6A,FRQ_Component,FRQ 6,KC.U1.09.10,KC.U1.09.quantitative_display_reading|KC.U7.11...,0.0,1,0,I,Rubric: identify the approximate numeric value...,I
19039,S025,MF2,27,MF2_FRQ6B,FRQ_Component,FRQ 6,KC.U7.12.13.14.15.16,KC.U7.12.mean_hypotheses|KC.U7.11.matched_pair...,1.0,1,100,E,Rubric: define the population mean paired diff...,E
19040,S025,MF2,27,MF2_FRQ6C,FRQ_Component,FRQ 6,KC.U7.12.13.14.15.16,KC.U7.13.t_test_conditions|KC.U7.11.matched_pa...,0.5,1,50,P,"Rubric: address paired structure, random sampl...",P
19041,S025,MF2,27,MF2_FRQ6D,FRQ_Component,FRQ 6,KC.U7.12.13.14.15.16,KC.U7.14.t_test_statistic_mean|KC.U7.15.p_valu...,1.0,1,100,E,"Rubric: use mean difference, standard error, d...",E


#### Same Modelling Code

In [46]:
import warnings
warnings.filterwarnings('ignore')

from pyBKT.models import Model
from sklearn.metrics import roc_auc_score

In [47]:
# Same Modelling Code

# Rename columns to what pyBKT expects

bkt_data = cleaned_2_data.rename(columns={
    'student_id':     'user_id',
    'primary_kc_id':  'skill_name',
    'observation_id': 'order_id',
    'score':          'correct',
})

# Keep only what pyBKT needs
bkt_data = bkt_data[['user_id', 'skill_name', 'correct', 'order_id']]

# Sort so each student's responses are in the right order
bkt_data = bkt_data.sort_values(['user_id', 'order_id']).reset_index(drop=True)
bkt_data['order_id'] = bkt_data.groupby('user_id').cumcount() + 1

print(bkt_data.head(10))
print(f"\nStudents: {bkt_data['user_id'].nunique()}")
print(f"KCs: {bkt_data['skill_name'].nunique()}")
print(f"Rows:{len(bkt_data)}")

  user_id                                skill_name  correct  order_id
0    S001      KC.U4.32.probability_from_simulation  0.00000         1
1    S001      KC.U4.32.probability_from_simulation  0.00000         2
2    S001  KC.U4.25.binomial_cumulative_probability  1.00000         3
3    S001                               KC.U4.23.24  0.00000         4
4    S001                               KC.U4.23.24  0.00000         5
5    S001                 KC.U4.26.binomial_mean_sd  0.00000         6
6    S001                               KC.U4.28.29  0.00000         7
7    S001                 KC.U4.26.binomial_mean_sd  0.00000         8
8    S001  KC.U4.25.binomial_cumulative_probability  0.00000         9
9    S001                               KC.U4.14.15  1.00000        10

Students: 25
KCs: 134
Rows:19043


In [48]:
bkt_data[bkt_data['user_id'] == 'S002'].head(40)

,user_id,skill_name,correct,order_id
780,S002,KC.U4.32.probability_from_simulation,1.00000,1
781,S002,KC.U4.32.probability_from_simulation,1.00000,2
782,S002,KC.U4.25.binomial_cumulative_probability,1.00000,3
783,S002,KC.U4.23.24,1.00000,4
784,S002,KC.U4.23.24,1.00000,5
785,S002,KC.U4.26.binomial_mean_sd,0.00000,6
786,S002,KC.U4.28.29,1.00000,7
787,S002,KC.U4.26.binomial_mean_sd,0.00000,8
788,S002,KC.U4.25.binomial_cumulative_probability,1.00000,9
789,S002,KC.U4.14.15,1.00000,10


In [49]:
# Fit the BKT model

model = Model(seed=42, num_fits=5)
model.fit(data=bkt_data)

params = model.params()
print(params)

                                                                value
skill                                         param   class          
KC.U4.32.probability_from_simulation          prior   default     NaN
                                              learns  default 1.00000
                                              guesses default 0.50000
                                              slips   default 0.50000
                                              forgets default 0.00000
...                                                               ...
KC.U2.04.categorical_association_independence prior   default     NaN
                                              learns  default 1.00000
                                              guesses default 0.50000
                                              slips   default 0.50000
                                              forgets default 0.00000

[670 rows x 1 columns]
